In [33]:
import pandas as pd
import os

def clean_hab_dataset(
    file_path,
    date_col="time",
    start_date="2017-01-01",
    drop_units_row=True,
    id_cols=None
):
    if id_cols is None:
        id_cols = ["location_code", "sampleid"]

    df = pd.read_csv(file_path)

    units_row = None
    if drop_units_row:
        units_row = df.iloc[0].copy()
        df = df.iloc[1:].copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    date_col = date_col.lower()
    id_cols = [col.lower() for col in id_cols]

    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in {file_path}")

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)

    for col in df.columns:
        if col != date_col and col not in id_cols:
            df[col] = pd.to_numeric(df[col], errors="ignore")

    df = df.dropna(axis=1, how="all").copy()
    df = df[df[date_col].notna()].copy()
    df = df.sort_values(date_col).reset_index(drop=True)

    df["sample_date"] = df[date_col].dt.date
    df["week_start"] = df[date_col].dt.to_period("W").dt.start_time
    df["year"] = df[date_col].dt.year
    df["month"] = df[date_col].dt.month

    df = df[df[date_col] >= pd.Timestamp(start_date, tz="UTC")].copy()

    return df, units_row

In [36]:
files = [
    "HABs-BodegaMarineLab_6cc8_2364_6bfa.csv",
    "HABs-BodegaMarineLabBuoy_e4a8_a9e8_5610.csv",
    "HABs-CalPolyPier_c282_7a86_708d.csv",
    "HABs-Humboldt_fe21_2e35_ece3.csv",
    "HABs-HumboldtSouthBay_676a_635f_2d92.csv",
    "HABs-InnerTomalesBay_1339_ee85_026b.csv",
    "HABs-MontereyWharf_1e75_40db_c5ad.csv",
    "HABs-NewportBeachPier_d635_dae7_af9c.csv",
    "HABs-SantaCruzWharf_86b0_cc60_0457.csv",
    "HABs-SantaMonicaPier_0b6e_b77e_5c4c.csv",
    "HABs-ScrippsPier_b18e_d2d9_23d2.csv",
    "HABs-StearnsWharf_eff4_30b3_6677.csv",
    "HABs-TomalesBayMid-ChannelBuoy_5b84_d3a1_5437.csv",
    "HABs-TomalesBayMouth_42e9_66dc_63e6.csv",
    "HABs-TrinidadPier_b69b_e00f_29f8.csv"
]

In [37]:
cleaned_data = {}
units_data = {}
summary_rows = []

for file in files:
    df_clean, units_row = clean_hab_dataset(file)

    dataset_name = os.path.splitext(file)[0]

    cleaned_data[dataset_name] = df_clean
    units_data[dataset_name] = units_row

    summary_rows.append({
        "dataset": dataset_name,
        "rows": len(df_clean),
        "cols": df_clean.shape[1],
        "start_date": df_clean["time"].min(),
        "end_date": df_clean["time"].max(),
        "duplicate_times": df_clean["time"].duplicated().sum()
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

C:\Users\isabe\AppData\Local\Temp\ipykernel_29016\509471700.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\isabe\AppData\Local\Temp\ipykernel_29016\509471700.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["week_start"] = df[date_col].dt.to_period("W").dt.start_time
C:\Users\isabe\AppData\Local\Temp\ipykernel_29016\509471700.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\isabe\AppData\Local\Temp\ipykernel_29016\509471700.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["week_start"] = df[date_col].dt.to_period("W").dt.s

,dataset,rows,cols,start_date,end_date,duplicate_times
0,HABs-BodegaMarineLab_6cc8_2364_6bfa,319,23,2020-01-20 20:47:00+00:00,2026-03-09 20:31:00+00:00,0
1,HABs-BodegaMarineLabBuoy_e4a8_a9e8_5610,53,23,2020-02-20 20:10:00+00:00,2026-01-28 19:08:00+00:00,0
2,HABs-CalPolyPier_c282_7a86_708d,481,40,2017-01-02 17:30:00+00:00,2026-04-05 17:32:00+00:00,0
3,HABs-Humboldt_fe21_2e35_ece3,100,12,2017-09-05 19:00:00+00:00,2026-01-12 20:30:00+00:00,3
4,HABs-HumboldtSouthBay_676a_635f_2d92,117,12,2017-08-16 19:00:00+00:00,2024-08-28 17:40:00+00:00,0
5,HABs-InnerTomalesBay_1339_ee85_026b,57,22,2021-01-29 19:38:00+00:00,2026-03-03 20:13:00+00:00,0
6,HABs-MontereyWharf_1e75_40db_c5ad,364,28,2017-01-04 16:15:00+00:00,2026-03-31 15:55:00+00:00,0
7,HABs-NewportBeachPier_d635_dae7_af9c,434,39,2017-01-02 18:40:00+00:00,2026-03-31 17:30:00+00:00,0
8,HABs-SantaCruzWharf_86b0_cc60_0457,452,23,2017-01-04 16:30:00+00:00,2026-03-25 16:00:00+00:00,0
9,HABs-SantaMonicaPier_0b6e_b77e_5c4c,481,29,2017-01-02 21:00:00+00:00,2026-03-30 21:00:00+00:00,0


In [38]:
output_folder = "cleaned_hab_data"
os.makedirs(output_folder, exist_ok=True)

for name, df in cleaned_data.items():
    clean_name = os.path.splitext(name)[0]
    save_path = os.path.join(output_folder, f"{clean_name}_cleaned.csv")
    df.to_csv(save_path, index=False)
    print("Saved:", save_path)

Saved: cleaned_hab_data\HABs-BodegaMarineLab_6cc8_2364_6bfa_cleaned.csv
Saved: cleaned_hab_data\HABs-BodegaMarineLabBuoy_e4a8_a9e8_5610_cleaned.csv
Saved: cleaned_hab_data\HABs-CalPolyPier_c282_7a86_708d_cleaned.csv
Saved: cleaned_hab_data\HABs-Humboldt_fe21_2e35_ece3_cleaned.csv
Saved: cleaned_hab_data\HABs-HumboldtSouthBay_676a_635f_2d92_cleaned.csv
Saved: cleaned_hab_data\HABs-InnerTomalesBay_1339_ee85_026b_cleaned.csv
Saved: cleaned_hab_data\HABs-MontereyWharf_1e75_40db_c5ad_cleaned.csv
Saved: cleaned_hab_data\HABs-NewportBeachPier_d635_dae7_af9c_cleaned.csv
Saved: cleaned_hab_data\HABs-SantaCruzWharf_86b0_cc60_0457_cleaned.csv
Saved: cleaned_hab_data\HABs-SantaMonicaPier_0b6e_b77e_5c4c_cleaned.csv
Saved: cleaned_hab_data\HABs-ScrippsPier_b18e_d2d9_23d2_cleaned.csv
Saved: cleaned_hab_data\HABs-StearnsWharf_eff4_30b3_6677_cleaned.csv
Saved: cleaned_hab_data\HABs-TomalesBayMid-ChannelBuoy_5b84_d3a1_5437_cleaned.csv
Saved: cleaned_hab_data\HABs-TomalesBayMouth_42e9_66dc_63e6_cleaned.c

## Potential Predictor Variables 

### 1. Physical Water Condiditons
These variables describe environmental conditions that influence algal growth:
- Temperature (`Temp`)
- Air Temperature (`Air_Temp`)
- Salinity (`Salinity`)
- Depth (`depth`)
- Location (`latitude`, `longitude`)

These are important because temperature and salinity directly affect biological processes and bloom formation.

### 2. Nutrient Concentrations
Nutrients are key drivers of algal growth:
- Phosphate (`Phosphate`)
- Silicate (`Silicate`)
- Nitrite (`Nitrite`)
- Nitrite + Nitrate (`Nitrite_Nitrate`)
- Ammonium (`Ammonium`)
- Nitrate (`Nitrate`)

High nutrient levels are often associated with increased bloom activity.

### 3. Chlorophyll and Productivity Indicators
These serve as proxies for overall phytoplankton biomass:
- Chlorophyll measurements (`Chl1`, `Chl2`, `Avg_Chloro`)
- Phaeophytin measurements (`Phaeo1`, `Phaeo2`, `Avg_Phaeo`)

These variables help indicate the presence and intensity of algal biomass in the water.

### 4. Species and Taxa Counts
These variables represent specific phytoplankton groups, including harmful species:
- *Pseudo-nitzschia* groups (`Pseudo_nitzschia_delicatissima_group`, `Pseudo_nitzschia_seriata_group`)
- Other taxa (`Alexandrium_spp`, `Dinophysis_spp`, `Prorocentrum_spp`, etc.)
- Total phytoplankton (`Total_Phytoplankton`)

These are especially useful because they directly measure organisms associated with HAB events.

### 5. Toxin Measurements (Potential Targets)
These variables may be used as response variables:
- Particulate domoic acid (`pDA`)
- Total domoic acid (`tDA`)
- Dissolved domoic acid (`dDA`)

These are strong indicators of harmful bloom severity.
